In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

FILE_LOCATION = '/kaggle/input/competitions/playground-series-s6e8/'
train_dataset = pd.read_csv(FILE_LOCATION + 'train.csv')
test_dataset = pd.read_csv(FILE_LOCATION + 'test.csv')
TARGET = train_dataset['addicted_label']
# train_dataset = train_dataset.drop(columns=['addicted_label', 'id','gender', 'academic_work_impact','age', 'stress_level'])
train_dataset = train_dataset.drop(columns=['addicted_label', 'id'])
y_id = test_dataset['id']
X_test = test_dataset.drop(columns=['id'])
# X_test = test_dataset.drop(columns=['id','gender', 'academic_work_impact','age', 'stress_level'])
N_SPLITS = 10
del test_dataset

/kaggle/input/competitions/playground-series-s6e8/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e8/train.csv
/kaggle/input/competitions/playground-series-s6e8/test.csv


In [2]:
%pip install catboost
import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score
from catboost import CatBoostClassifier

use_device = 'cuda' if torch.cuda.is_available() else 'cpu'

# TE_COLS = ['daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
#            'work_study_hours', 'sleep_hours', 'notifications_per_day',
#            'app_opens_per_day', 'weekend_screen_time']

TE_COLS = train_dataset.columns.tolist()

def to_levels(df, cols):
    """Convert columns to string labels for CatBoost's cat_features.
    NaN becomes an explicit '__missing__' level."""
    return pd.DataFrame({
        c: df[c].astype(object).fillna("__missing__").astype(str).values
        for c in cols
    }, index=df.index)

def build_cat_frame(X, cols):
    """Numeric block (unchanged) + raw string levels of `cols`, prefixed
    'lvl_', so CatBoost can treat them as categorical and build its own
    target statistics internally."""
    str_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
    X = X.copy()
    if str_cols:
        X[str_cols] = to_levels(X, str_cols)          # <-- added: NaN -> "__missing__"
    lvl_block = to_levels(X, cols).add_prefix("lvl_")
    out = pd.concat([X.reset_index(drop=True), lvl_block.reset_index(drop=True)], axis=1)
    cat_cols = str_cols + lvl_block.columns.tolist()
    cat_feature_idx = [out.columns.get_loc(c) for c in cat_cols]
    return out, cat_feature_idx
    
X_full = train_dataset
y_full = TARGET
X_test = X_test[X_full.columns]
X_test_reset = X_test.reset_index(drop=True)

oof_predictions = np.zeros(len(X_full))

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=48)

fold_roc_aucs = []
fold_accuracies = []
test_probas_folds = []

for fold_num, (train_idx, val_idx) in enumerate(cv.split(X_full, y_full), start=1):
    X_tr = X_full.iloc[train_idx].reset_index(drop=True)
    X_va = X_full.iloc[val_idx].reset_index(drop=True)
    y_tr = y_full.iloc[train_idx].reset_index(drop=True)
    y_va = y_full.iloc[val_idx].reset_index(drop=True)

    X_tr_cat, cat_idx = build_cat_frame(X_tr, TE_COLS)
    X_va_cat, _ = build_cat_frame(X_va, TE_COLS)
    X_test_cat, _ = build_cat_frame(X_test_reset, TE_COLS)

    model = CatBoostClassifier(
        iterations=20000,
        learning_rate=0.03,
        depth=6,
        eval_metric='AUC',
        early_stopping_rounds=100,
        random_seed=48,
        verbose=0,
        **({"task_type": "GPU"} if use_device == "cuda" else {})
    )
    model.fit(X_tr_cat, y_tr, eval_set=(X_va_cat, y_va), cat_features=cat_idx, verbose=0)

    y_proba = model.predict_proba(X_va_cat)[:, 1]
    oof_predictions[val_idx] = y_proba
    y_pred = model.predict(X_va_cat)

    fold_auc = roc_auc_score(y_va, y_proba)
    fold_acc = accuracy_score(y_va, y_pred)
    fold_roc_aucs.append(fold_auc)
    fold_accuracies.append(fold_acc)
    test_probas_folds.append(model.predict_proba(X_test_cat)[:, 1])

    print(f"Fold {fold_num}: ROC-AUC = {fold_auc:.6f}, Accuracy = {fold_acc:.6f}, "
          f"best_iteration = {model.get_best_iteration()}")

fold_roc_aucs = np.array(fold_roc_aucs)
fold_accuracies = np.array(fold_accuracies)

print(f"\nMean ROC-AUC: {fold_roc_aucs.mean():.6f}  |  Std: {fold_roc_aucs.std():.6f}")
print(f"Mean Accuracy: {fold_accuracies.mean():.6f}  |  Std: {fold_accuracies.std():.6f}")

Note: you may need to restart the kernel to use updated packages.


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 1: ROC-AUC = 0.967948, Accuracy = 0.908934, best_iteration = 8149


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 2: ROC-AUC = 0.966911, Accuracy = 0.906837, best_iteration = 9567


...

Fold 8: ROC-AUC = 0.968585, Accuracy = 0.910048, best_iteration = 9507


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 9: ROC-AUC = 0.967979, Accuracy = 0.909195, best_iteration = 8815


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 10: ROC-AUC = 0.967701, Accuracy = 0.909642, best_iteration = 8779

Mean ROC-AUC: 0.967467  |  Std: 0.000684
Mean Accuracy: 0.908286  |  Std: 0.001114


In [3]:
# Submission: average predictions across all 10 fold models
test_probabilities = np.mean(test_probas_folds, axis=0)

submission = pd.DataFrame({
    'id': y_id,
    'addicted_label': test_probabilities
})
submission.to_csv('catboost_submission.csv', index=False)

oof_df = pd.DataFrame({
    'id': X_full['id'] if 'id' in X_full.columns else np.arange(len(X_full)),
    'true_label': y_full.values,
    'oof_proba': oof_predictions
})
oof_df.to_csv('catboost_oof.csv', index=False)

test_probs_df = pd.DataFrame({
    'id': y_id,
    'test_proba': test_probabilities
})
test_probs_df.to_csv('catboost_test_probs.csv', index=False)

print(f"Full OOF ROC-AUC: {roc_auc_score(y_full, oof_predictions):.6f}")
print(submission.head())

Full OOF ROC-AUC: 0.967466
       id  addicted_label
0  691369        0.999629
1  691370        0.969124
2  691371        0.813343
3  691372        0.997463
4  691373        0.999076
